## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
fatal: unable to access 'https://github.com/Lv1g1/RecSys-Challenge-2025.git/': Could not resolve host: github.com


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [ ]:
import pandas as pd
import os
import optuna
from xgboost import XGBRanker
from sklearn.metrics import roc_auc_score
import gc

from Challenge.paths import load_xgboost_cv_folds, XGBOOST_DATAFRAMES
from Challenge.hyper_tuning import ModelOptimizer
from Challenge.utils import evaluate_recommender
from Challenge.XGBoostReranker import XGBoostRerankerRecommender

Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


## **Load Data**

In [4]:
_, URM_outer, _ = load_xgboost_cv_folds()
del _

In [5]:
file_path = os.path.join(XGBOOST_DATAFRAMES, "training_data.parquet")

X_train = pd.read_parquet(
    file_path,
    engine='fastparquet'
)

print(f"Loaded successfully from: {file_path}")
X_train

Loaded successfully from: /home/luigi/RecSys/xg_boost_data/dataframes/training_data.parquet


,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,ItemKNN_jaccard_Recommended,ItemKNN_asymmetric_Score,ItemKNN_asymmetric_RankPosition,ItemKNN_asymmetric_Recommended,ItemKNN_tversky_Score,ItemKNN_tversky_RankPosition,ItemKNN_tversky_Recommended,ItemKNN_dice_Score,ItemKNN_dice_RankPosition,ItemKNN_dice_Recommended,UserKNN_cosine_Score,UserKNN_cosine_RankPosition,UserKNN_cosine_Recommended,UserKNN_jaccard_Score,UserKNN_jaccard_RankPosition,UserKNN_jaccard_Recommended,UserKNN_asymmetric_Score,UserKNN_asymmetric_RankPosition,UserKNN_asymmetric_Recommended,UserKNN_tversky_Score,UserKNN_tversky_RankPosition,UserKNN_tversky_Recommended,UserKNN_dice_Score,UserKNN_dice_RankPosition,UserKNN_dice_Recommended,SLIMElasticNet_Score,SLIMElasticNet_RankPosition,SLIMElasticNet_Recommended,EASE_R_Score,...,P3alpha_RankPosition,P3alpha_Recommended,RP3beta_Score,RP3beta_RankPosition,RP3beta_Recommended,MatrixFactorization_WARP_Score,MatrixFactorization_WARP_RankPosition,MatrixFactorization_WARP_Recommended,MatrixFactorization_BPR_Score,MatrixFactorization_BPR_RankPosition,MatrixFactorization_BPR_Recommended,MatrixFactorization_SVDpp_Score,MatrixFactorization_SVDpp_RankPosition,MatrixFactorization_SVDpp_Recommended,SLIM_BPR_Score,SLIM_BPR_RankPosition,SLIM_BPR_Recommended,NMF_Score,NMF_RankPosition,NMF_Recommended,MultVAE_Score,MultVAE_RankPosition,MultVAE_Recommended,ItemKNN_tversky_MaxSim,ItemKNN_tversky_StdSim,RP3beta_MaxSim,RP3beta_StdSim,SLIMElasticNet_MaxSim,SLIMElasticNet_StdSim,Counter_Recommended,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,5609,False,0.270185,116,0,0.428426,133,0,0.415858,18,1,0.260064,20,0,0.266434,25,0,0.403760,30,0,0.089980,244,0,0.149958,110,0,0.132762,164,0,0.159860,129,0,0.156226,92,0,0.246855,29,0,0.205206,...,15,1,0.176334,67,0,-0.290388,6223,0,0.618150,113,0,0.919342,757,0,0.434787,54,0,3.164229e-01,6,1,0.502151,213,0,0.170862,0.022241,0.026979,0.003512,0.047947,0.008360,3,410.333344,1341.456177,4.480386,20.308155,0.292546,0.234324,0.334969,2.938539,58,2319
1,0,1940,False,0.612331,7,1,0.632690,26,0,0.494830,18,1,0.000000,685,0,0.000000,532,0,0.668248,9,1,0.496413,3,1,0.391590,8,1,0.526911,3,1,0.464513,7,1,0.420787,8,1,0.294995,31,0,0.230905,...,1398,0,0.000000,953,0,-0.154930,5213,0,0.758786,4,1,0.947563,929,0,0.812230,1,1,5.687535e-23,5530,0,0.940704,0,1,0.000000,0.000000,0.000000,0.000000,0.036569,0.005896,11,735.333313,1595.107910,2.655794,6.240859,0.406598,0.335242,-0.084116,-1.069763,60,5214
2,0,3258,True,0.078685,762,0,0.342990,255,0,0.113605,205,0,0.479622,19,1,0.352871,18,1,0.119971,249,0,0.146184,119,0,0.139487,124,0,0.186709,87,0,0.208194,86,0,0.142575,129,0,0.299599,30,0,0.271565,...,32,0,0.358544,22,0,0.025339,3217,0,0.332861,423,0,0.965392,264,0,0.035074,461,0,1.446359e-01,45,0,0.621546,120,0,0.000000,0.000000,0.000000,0.000000,0.043448,0.006414,2,319.952393,688.967896,4.085459,17.637278,0.266451,0.218058,1.866132,4.445840,60,670
3,0,1504,True,0.013036,2678,0,0.349408,241,0,0.153183,140,0,0.760277,7,1,0.427471,12,1,0.210104,120,0,0.140780,128,0,0.093872,241,0,0.173824,101,0,0.143546,190,0,0.081581,315,0,0.312683,29,0,0.275832,...,50,0,0.368972,20,0,0.058512,2834,0,0.052843,1667,0,0.869248,2935,0,0.013861,659,0,2.249700e-02,449,0,0.090613,1312,0,0.218544,0.033658,0.083940,0.012741,0.119502,0.017147,2,675.000000,992.564087,1.595576,1.127617,0.228266,0.229798,1.721957,2.795686,60,111
4,0,3754,False,0.070346,845,0,0.350854,236,0,0.162925,129,0,0.318583,39,0,0.248214,33,0,0.150988,190,0,0.126851,155,0,0.114215,186,0,0.128566,169,0,0.114186,278,0,0.121005,177,0,0.323561,28,0,0.314643,...,95,0,0.200135,57,0,-0.110207,4761,0,0.241049,694,0,0.961626,379,0,0.076845,313,0,2.849000e-02,348,0,0.398853,417,0,0.401474,0.051396,0.092920,0.011896,0.2

In [6]:
if not X_train['UserID'].is_monotonic_increasing:
    print("⚠️ Data is NOT sorted by UserID.")
    print("Sorting and overwriting file for future efficiency...")
    
    # Sort and reset index (critical for contiguous memory in ML)
    X_train = X_train.sort_values(by='UserID').reset_index(drop=True)
    
    # Overwrite the file on disk
    X_train.to_parquet(
        path=file_path,
        engine='fastparquet',
        compression='zstd',
        index=False
    )
    
    print(f"✅ Data sorted and saved to {file_path}")

In [7]:
assert X_train['UserID'].is_monotonic_increasing, "UserID column is not sorted in increasing order!"

# Group Calculation
groups = X_train.groupby("UserID").size().values

# Target and Feature Matrix Definition
y_train = X_train["Label"]

# Drop the identifiers (UserID, ItemID) and the target (Label), keeping only the features
X_train = X_train.drop(columns=["Label", "UserID", "ItemID"], errors='ignore') 

assert groups.sum() == X_train.shape[0], "Group sum does not match total data size!"

In [8]:
file_path = os.path.join(XGBOOST_DATAFRAMES, "validation_data.parquet")

X_val = pd.read_parquet(
    file_path,
    engine='fastparquet'
)

print(f"Loaded successfully from: {file_path}")
X_val

Loaded successfully from: /home/luigi/RecSys/xg_boost_data/dataframes/validation_data.parquet


,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,ItemKNN_jaccard_Recommended,ItemKNN_asymmetric_Score,ItemKNN_asymmetric_RankPosition,ItemKNN_asymmetric_Recommended,ItemKNN_tversky_Score,ItemKNN_tversky_RankPosition,ItemKNN_tversky_Recommended,ItemKNN_dice_Score,ItemKNN_dice_RankPosition,ItemKNN_dice_Recommended,UserKNN_cosine_Score,UserKNN_cosine_RankPosition,UserKNN_cosine_Recommended,UserKNN_jaccard_Score,UserKNN_jaccard_RankPosition,UserKNN_jaccard_Recommended,UserKNN_asymmetric_Score,UserKNN_asymmetric_RankPosition,UserKNN_asymmetric_Recommended,UserKNN_tversky_Score,UserKNN_tversky_RankPosition,UserKNN_tversky_Recommended,UserKNN_dice_Score,UserKNN_dice_RankPosition,UserKNN_dice_Recommended,SLIMElasticNet_Score,SLIMElasticNet_RankPosition,SLIMElasticNet_Recommended,EASE_R_Score,...,P3alpha_RankPosition,P3alpha_Recommended,RP3beta_Score,RP3beta_RankPosition,RP3beta_Recommended,MatrixFactorization_WARP_Score,MatrixFactorization_WARP_RankPosition,MatrixFactorization_WARP_Recommended,MatrixFactorization_BPR_Score,MatrixFactorization_BPR_RankPosition,MatrixFactorization_BPR_Recommended,MatrixFactorization_SVDpp_Score,MatrixFactorization_SVDpp_RankPosition,MatrixFactorization_SVDpp_Recommended,SLIM_BPR_Score,SLIM_BPR_RankPosition,SLIM_BPR_Recommended,NMF_Score,NMF_RankPosition,NMF_Recommended,MultVAE_Score,MultVAE_RankPosition,MultVAE_Recommended,ItemKNN_tversky_MaxSim,ItemKNN_tversky_StdSim,RP3beta_MaxSim,RP3beta_StdSim,SLIMElasticNet_MaxSim,SLIMElasticNet_StdSim,Counter_Recommended,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,6411,False,0.081851,717,0,0.701361,5,1,0.681310,8,1,0.513640,3,1,0.274786,18,1,0.633201,12,1,0.435580,7,1,0.433953,2,1,0.453117,6,1,0.465966,6,1,0.448846,2,1,0.371053,11,1,0.369707,...,1,1,0.426960,3,1,-0.387449,6377,0,0.829961,4,1,0.942578,1194,0,0.226819,149,0,4.061391e-01,3,1,0.714231,14,1,0.508856,0.092755,0.084230,0.014055,0.193600,0.029008,17,407.142853,1399.072754,4.281490,18.897297,0.447156,0.278489,-1.061027,3.246427,80,874
1,0,2168,False,0.078760,750,0,0.540989,40,0,0.491260,19,1,0.341257,11,1,0.433244,9,1,0.460370,31,0,0.212754,65,0,0.173101,87,0,0.200741,82,0,0.260747,54,0,0.194136,75,0,0.434087,10,1,0.346655,...,10,1,0.353215,7,1,-0.065467,4105,0,0.619865,59,0,0.928368,1642,0,0.114839,251,0,1.281659e-01,64,0,0.540388,137,0,0.531242,0.081012,0.107411,0.015765,0.340882,0.046237,7,358.047607,936.414124,3.650616,14.062557,0.333984,0.222666,0.733968,1.210770,80,841
2,0,6098,True,0.082225,713,0,0.708915,3,1,0.740307,7,1,0.532180,2,1,0.535007,4,1,0.738922,7,1,0.436021,6,1,0.411771,4,1,0.454141,5,1,0.474621,5,1,0.429967,4,1,0.434650,9,1,0.376617,...,0,1,0.438815,2,1,0.334395,783,0,0.779634,12,1,0.949798,862,0,0.206347,168,0,4.079978e-01,2,1,0.716227,12,1,0.519439,0.094345,0.087893,0.014549,0.218410,0.032361,17,124.571426,279.955811,2.169494,3.165847,0.503251,0.205144,0.271329,0.168902,80,878
3,0,3049,False,0.128114,412,0,0.693846,6,1,0.457670,22,0,0.195068,34,0,0.388641,10,1,0.429929,39,0,0.146193,139,0,0.122195,161,0,0.147806,147,0,0.217557,91,0,0.139562,144,0,0.439321,8,1,0.373708,...,63,0,0.192713,32,0,-0.079397,4266,0,0.612629,63,0,0.952464,712,0,0.126273,232,0,1.848137e-01,29,0,0.552071,126,0,0.544770,0.060525,0.096697,0.010743,0.449787,0.050070,4,321.095245,918.941772,4.353480,19.441338,0.310892,0.245103,0.985542,0.890398,80,1368
4,0,2399,False,0.442124,27,0,0.639774,15,1,0.261722,82,0,0.254043,16,1,0.529372,5,1,0.435068,36,0,0.350725,17,1,0.310341,18,1,0.347975,16,1,0.336895,24,0,0.331354,16,1,0.494412,7,1,0.452367,...,8,1,0.305896,10,1,-0.467737,6590,0,0.770921,13,1,0.953759,652,0,0.589067,15,1,2.287863e-01,19,1,0.681651,25,0,0.000000,0.000000,0.038185,0.004242,0.058481,0.010122,14,362.523804,1433.629395,4.515592,20.544024,

In [ ]:
recommender = XGBoostRerankerRecommender(
    XGB_model=None,
    df=X_val
)

## **Optimize XG Boost hypeparameters**

In [9]:
optimizer = ModelOptimizer("XG_Boost_v1")

In [10]:
STUDY_NAME = XGBoostRerankerRecommender.RECOMMENDER_NAME + "_v1_1"

In [ ]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        'learning_rate': optuna_trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'n_estimators': optuna_trial.suggest_int('n_estimators', 20, 1000),
        'max_depth': optuna_trial.suggest_int('max_depth', 3, 10),
        'reg_alpha': optuna_trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': optuna_trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'colsample_bytree': optuna_trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'subsample': optuna_trial.suggest_float('subsample', 0.6, 1.0),
        
        # You can tune the objective too, but pairwise is usually standard for ranking
        'objective': optuna_trial.suggest_categorical('objective', ['rank:pairwise', 'rank:ndcg']),
        'grow_policy': 'depthwise' # Keep this fixed or tune ['depthwise', 'lossguide']
    }

    # Instantiate XGB model with current hyperparameters
    XGB_model = XGBRanker(
        objective=params['objective'],
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        max_depth=params['max_depth'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        colsample_bytree=params['colsample_bytree'],
        subsample=params['subsample'],
        grow_policy=params['grow_policy'],
        n_jobs=-1,
        random_state=42,
        verbosity=0 # Silence output during tuning
    )

    # Fit XG boost reranker
    XGB_model.fit(
        X_train,
        y_train,
        group=groups,
        verbose=True
    )

    # Evaluate training performance
    y_pred_scores = XGB_model.predict(X_train)
    auc_score = roc_auc_score(y_train, y_pred_scores)
    print(f"Training AUC Score: {auc_score:.4f}")

    # Set the XGB model in the recommender
    recommender.XGB_model = XGB_model
    
    # Evaluate
    score = evaluate_recommender(recommender, at=20, URM_validation=URM_outer)
    print(f"Validation Score (RECALL@20): {score:.4f}")

    optimizer.log_folds([score], params)

    return score

In [12]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-27 20:31:54,669] A new study created in RDB with name: XGBoostRerankerRecommender_v1_1


  0%|          | 0/100 [00:00<?, ?it/s]

Training AUC Score: 0.7894
[W 2025-11-27 20:49:29,179] Trial 0 failed with parameters: {'learning_rate': 0.01222982191393915, 'n_estimators': 567, 'max_depth': 3, 'reg_alpha': 0.7709208908307081, 'reg_lambda': 0.41703497767977304, 'colsample_bytree': 0.7895213800179919, 'subsample': 0.9730498550868242, 'objective': 'rank:ndcg'} because of the following error: UnboundLocalError("cannot access local variable 'df' where it is not associated with a value").
Traceback (most recent call last):
  File "/home/luigi/.venvs/recsys/lib/python3.13/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_30608/342212902.py", line 50, in objective_function
    if df is None or index_df is None or user_map is None:
       ^^
UnboundLocalError: cannot access local variable 'df' where it is not associated with a value
[W 2025-11-27 20:49:29,195] Trial 0 failed with value None.


UnboundLocalError: cannot access local variable 'df' where it is not associated with a value

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Change ranges of Optimization**

In [ ]:
STUDY_NAME = XGBoostRerankerRecommender.RECOMMENDER_NAME + "_v1_2"

In [ ]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:    
    params = {
        'learning_rate': optuna_trial.suggest_float('learning_rate', 1e-3, 0.2, log=True),
        'n_estimators': optuna_trial.suggest_int('n_estimators', 20, 1000),
        'max_depth': optuna_trial.suggest_int('max_depth', 4, 14),
        'reg_alpha': optuna_trial.suggest_float('reg_alpha', 1e-3, 100.0, log=True),
        'reg_lambda': optuna_trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'colsample_bytree': optuna_trial.suggest_float('colsample_bytree', 0.4, 0.9),
        'subsample': optuna_trial.suggest_float('subsample', 0.6, 1.0),
        
        'objective': 'rank:ndcg',
        'grow_policy': 'depthwise'
    }

    # Instantiate XGB model with current hyperparameters
    XGB_model = XGBRanker(
        objective=params['objective'],
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        max_depth=params['max_depth'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        colsample_bytree=params['colsample_bytree'],
        subsample=params['subsample'],
        grow_policy=params['grow_policy'],
        n_jobs=-1,
        random_state=42,
        verbosity=0
    )

    # Fit XG boost reranker
    XGB_model.fit(
        X_train,
        y_train,
        group=groups,
        verbose=True
    )

    # Evaluate training performance
    y_pred_scores = XGB_model.predict(X_train)
    auc_score = roc_auc_score(y_train, y_pred_scores)
    print(f"Training AUC Score: {auc_score:.4f}")

    # Instantiate the recommender with the trained XGB model
    recommender_instance = XGBoostRerankerRecommender(
        XGB_model=XGB_model,
        df=X_val
    )
    
    # Evaluate
    score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_val)
    print(f"Validation Score (RECALL@20): {score:.4f}")

    optimizer.log_folds([score], params)

    return score

In [ ]:
optuna_study = optimizer.create_study(STUDY_NAME, pruner=None)

In [ ]:
known_best_params = {
    'learning_rate': 0.0333,
    'n_estimators': 456, 
    'max_depth': 10,
    'reg_alpha': 9.81,
    'reg_lambda': 0.006,
    'colsample_bytree': 0.59,
    'subsample': 0.82
}

# Enqueue the known best parameters as a trial
optuna_study.enqueue_trial(known_best_params)

In [ ]:
optimizer.optimize(
    objective_function=objective_function,
    n_trials=100
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)